# After teacher training, can the rest of the fine-tune go into Δ and stay faithful?

*E43, `experiments/teacher_training/remainder.py`, 2026-10-28. Record:
`reports/analyses/2026-10-28_analysis_e43-teacher-remainder.md`. Plan: `reports/plans/2026-10-28_plan_e43-teacher-remainder.md`.
Starts from E41's teacher-trained states; figures from `experiments/teacher_training/plot_e43.py`.*

## The answer

No. We took the decomposition after teacher training and put the part of the fine-tune it had not reproduced into Δ, so
the weights equal the fine-tuned model's exactly. The adversarial check then failed at both steps we tried. It read 2.5 and
1.9 times teacher training's own ratio, and worse than leaving the whole fine-tune in Δ. On plain text, putting Δ′ back
to Δ loses a quarter to two-fifths of the fine-tune's behaviour, the gap teacher training's own state already has
there. Teacher training's own state, with Δ untouched, stays the
map.

## What we were trying to test

The decomposition writes each weight matrix as $W = U V^\top + \Delta$: the alive subcomponents' write and read vectors,
plus a residual Δ. Γ decides which subcomponents matter on each input. **Teacher training** trains only $U$ and $V$ so
that the model built from them imitates the fine-tuned model, with Γ and Δ frozen. It keeps the behaviour and, on
finished fine-tunes, passes the adversarial check. But its weights are not the fine-tuned model's.

The user asked whether the gap can be closed afterwards. Per matrix, the **remainder** is $R = W_{\text{ft}} - W_{\text{tt}}$,
the fine-tuned weights minus the weights after teacher training. **Teacher training plus remainder** keeps the trained
$U'$ and $V'$ and sets $\Delta' = \Delta + R$. Its weights now equal the fine-tuned model's exactly. For this experiment
only, the user read "Δ does not absorb the update" by behaviour: Δ′ passes if putting it back to Δ barely changes the
outputs.

The fine-tune is E41's: Adam on the emoticon task, where an emoticon is followed by `o`, stopped at step 10 and at step
1,000. Each step gets four states, read on the same rows in one job on one H100:

```
do-nothing            subcomponents untouched, the whole fine-tune in Δ          the positive control
least-squares split   the fine-tune split over the subcomponents, the rest in Δ  exact weights, fails at step 10 (E22)
teacher training      trained subcomponents, Δ untouched                         the map so far
plus remainder        trained subcomponents, the remainder in Δ                  exact weights, new here
```

Each state gets two readings:

- **The adversarial check.** A 20-step mask adversary searches for the masks that most change the outputs on 256 Pile
  rows. Its KL over the same search on the unchanged model is the ratio; 1 means no change. This job's lines are pass at
  or below 1.40 and fail at or above 1.61, set from the unchanged model's spread over three adversary seeds.
- **The removal share.** Put Δ′ back to Δ, then measure $\mathrm{KL}(\text{fine-tune} \,\|\, \text{that model})$ over the same
  KL for the unchanged model. 0 means Δ′ carried nothing the fine-tune does; 1 means it carried all of it. Do-nothing
  reads 1 by construction. It is read on held-out task windows and on the Pile rows. Teacher training's own state lands
  on the same model when Δ′ goes back, so it shares plus remainder's value.

## What we expected, and why

- **Task text: almost nothing in Δ′.** Putting Δ′ back gives teacher training's own state, which E41 measured close to
  the fine-tune on task windows. This was a check that the run reproduced E41, not a test.
- **Plain text: between inert and carrying.** We expected the removal share on Pile rows above 0.2 and below 0.5, at
  medium confidence. Teacher training on task text, on another task, read about 0.3, so teacher training's own
  state differs from the fine-tuned model off the task.
- **The check: no worse than a quarter above teacher training's own ratio.** We leaned that it holds, with high
  confidence at step 1,000 and medium at step 10. The claim was that the remainder is inert, so the adversary would not
  find it. The null was that the remainder carries what teacher training got wrong off task text, and the adversary reads it as
  it reads do-nothing's Δ.

## Does teacher training plus remainder pass the check as teacher training's own state does?

![adversarial ratio of the four states at steps 10 and 1000](../experiments/teacher_training/figures/e43_check.png)

**Reading it.** No, at either step, and by far more than the noise. The orange diamonds are teacher training plus
remainder. At step 10 it reads 3.92, two and a half times teacher training's own 1.55 in blue. At step 1,000 it reads
1.87, about twice teacher training's own 0.96, which itself passes. In both cases it sits above the grey square, the fine-tune left whole
in Δ, which reads 2.52 and 1.50. So putting the remainder into Δ undoes everything teacher training gained on the check,
and more. Only the least-squares split, in purple, reads worse. The noise is small beside this: the unchanged model's
adversary spread is under 8%. Teacher training's spread across batch orders is 21% at step 10 and 3% at step 1,000. The
stochastic-mask ratio stays near 1 for every state, so the failure is the adversary's alone.

## How much of the fine-tune's behaviour sits in Δ′?

![removal share on task text and on Pile text for the states](../experiments/teacher_training/figures/e43_removal.png)

**Reading it.** On task text almost none; on plain text a quarter to two-fifths. On the left, putting Δ′ back leaves
0.001 of the unchanged model's distance to the fine-tune at step 10 and 0.006 at step 1,000. These match E41's copies to
the digit, so the run reproduced them. On the right, on Pile rows, the same removal leaves 0.24 at step 10 and 0.42 at
step 1,000, between the inert and carrying lines. The least-squares split leaves less, 0.12 and 0.10. The share is a
ratio over a small number: the unchanged model is only 0.02 and 0.06 nats from the fine-tune on these rows.

## How large is the remainder?

![weight in Δ′ beyond Δ, as a share of the fine-tune's update, for the four states](../experiments/teacher_training/figures/e43_delta.png)

**Reading it.** At step 10 the remainder is far larger than the fine-tune itself. The orange diamond reads 5.1: the
squared size of the remainder, over all 24 matrices, is five times the fine-tune's. The plan expected about 1. Its cosine
with teacher training's own weight change is −0.90, so at step 10 the remainder mostly undoes that change. At step 1,000 the
remainder is about the fine-tune's size, 1.1, with cosine −0.42. This is a reported number, not a filed prediction. It
says teacher training reaches the fine-tune's behaviour with a weight change that points elsewhere, and the remainder has to
take that back.

In [1]:
# The four states at each step, from the results file (no model).
import json
from pathlib import Path

r = json.loads(Path("../experiments/teacher_training/results/teacher_training_e43.json").read_text())
names = {"nothing": "do-nothing", "least_squares": "least-squares split",
         "teacher_training": "teacher training", "plus_remainder": "plus remainder"}
print(f"{'state':22s} {'step':>5s} {'check':>6s} {'Pile removal':>13s} {'weight in Δ′':>13s}")
for k in ("10", "1000"):
    for key, name in names.items():
        s = r["steps"][k][key]
        b = s.get("b_pile", r["steps"][k]["plus_remainder"]["b_pile"])
        print(f"{name:22s} {k:>5s} {s['pgd']:6.2f} {b:13.3f} {s['delta_share']:13.2g}")

state                   step  check  Pile removal  weight in Δ′
do-nothing                10   2.52         1.000             1
least-squares split       10   6.96         0.125          0.36
teacher training          10   1.55         0.235       2.3e-09
plus remainder            10   3.92         0.235           5.1
do-nothing              1000   1.50         1.000             1
least-squares split     1000   2.11         0.099          0.38
teacher training        1000   0.96         0.417       1.3e-09
plus remainder          1000   1.87         0.417           1.1


## What we learned

Keeping the fine-tuned model's exact weights by adding the remainder to Δ costs the faithfulness teacher training had.
On this emoticon fine-tune, at both steps, the decomposition with the remainder in Δ fails the adversarial check and reads
worse than doing nothing. For the weak version in `GOALS.md`, the map stays teacher training's own state, with Δ
untouched and weights that are not the fine-tune's. The exact-weights route is closed in this form.

## How compelling is the evidence

- **The miss is large against every noise source we have.** Plus remainder read 2.5 and 1.9 times teacher training's
  ratio, against a filed allowance of 1.25. The unchanged model's adversary spread is 7.5%. Teacher training's batch-order spread
  is 21% at step 10 and 3% at step 1,000.
- **The comparison is paired.** Same fine-tune, same teacher-trained subcomponents, same rows, same adversary seed, so their noise is shared.
- **Every control reproduced.** The unchanged model's floor, the fine-tunes, do-nothing and teacher training's own states all matched E41
  on this card to four digits or better.
- **One draw.** One task, one fine-tune seed, one batch order, one adversary start of 20 steps. The state's adversary
  KL is a lower bound, so more starts could only raise it; they would also raise the unchanged model's reading that
  the ratio divides by.
- **The removal share sees only the 256 Pile rows and the task windows.** It sees nothing at the trigger alone.

## Surprises

- **Something we did not expect: the remainder in Δ made the check fail, worse than doing nothing.** We predicted it
  would stay within a quarter of teacher training's own ratio, leaning high at step 1,000. It read 2.5 and 1.9 times that
  ratio, and above the fine-tune left whole in Δ. The run did not test why. It did not locate which part of
  Δ′ the adversary uses.
- **The remainder at step 10 is five times the fine-tune's size.** The plan expected about the same size. It points
  mostly opposite to teacher training's weight change.
- None on task text: the removal share reproduced E41 to the digit.

## What next

1. **Accept, and keep teacher training's own state as the map.** Recommended, with no run: the plan's tree says no run
   for this outcome. A result that would reopen it: a version of the remainder that passes the check at step 1,000.
2. **Run teacher training on Pile text, then add its remainder.** Teacher training on Pile text came closer to
   the fine-tuned model on plain text on another task. If its plus-remainder state passed, the exact-weights map would reopen. It targets the
   plain-text share; nothing yet says it would pass the check. Needs a new plan.
3. **Split the remainder over the subcomponents by least squares first.** Only what is left would go to Δ. The
   least-squares split of the whole fine-tune reads worse still, so we expect little. Needs a new plan.

## Scorecard

Filed in `experiments/teacher_training/results/predictions_e43_v2.json` before the run, sha256 `204c1fbb…`, unedited.
Scored by the run's `score` block and checked against the filed rules.

| id | claim | rule | read, step 10 / 1000 | verdict |
|---|---|---|---|---|
| C1, G1, PC, G3, G2 | controls reproduce E41 | as filed | floor 0.0748, all to four digits; G2 4.1e-5 / 4.3e-5 | pass |
| P1 | task text: Δ′ carries almost nothing | removal share ≤ 0.05, teacher training's P ≥ 0.9 | 0.0013 / 0.0056; P 0.9994 / 0.9968 | true, unscored |
| P2 | plain text: Δ′ inert | ≤ 0.2 inert, ≥ 0.5 carries | 0.235 / 0.417 | between, as leaned |
| P3 | check within 1.25× teacher training's own | ratio ≤ 1.25 × own | 3.92 / 1.87 against 1.55 / 0.96; 2.54× / 1.94× | does not hold |

## Not run

- The broad lowercase fine-tune; step 100; other tasks; SGD and LoRA teachers; teacher training on Pile text; Qwen.
- Batch-order or fine-tune seed repeats; adversary seeds 1 and 2; a longer adversary.
- The least-squares split of the remainder into the subcomponents; any test of which part of Δ′ the adversary uses.
- Saving states: none were saved; the analysis reads only the results file.

**Provenance.** Results `experiments/teacher_training/results/teacher_training_e43.json`; log
`experiments/teacher_training/results/large/run_e43.log`. RunPod NVIDIA H100 80GB HBM3; the job ran 16:26 to 16:38 on
2026-10-28, about 12 minutes. Cost about $2.5: session spend $0.11 to $2.68, including about 12 minutes of
set-up, against the plan's estimate of about $2.